<a href="https://colab.research.google.com/github/rafagmoraes-dot/dashboard-parques-curitiba/blob/main/C%C3%B3pia_de_Aula04_Exerc%C3%ADcio_Rafaela_Moraes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install geopandas folium

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import geopandas as gpd

caminho = '/content/drive/MyDrive/Colab Notebooks/Aula04/DIVISA_DE_BAIRROS_SIRGAS/DIVISA_DE_BAIRROS.shp'

bairros = gpd.read_file(caminho)

bairros.head()

,OBJECTID,CODIGO,TIPO,NOME,FONTE,CD_REGIONA,NM_REGIONA,SHAPE_AREA,SHAPE_LEN,geometry
0,111,29.0,DIVISA DE BAIRROS,SEMINÁRIO,Decreto Municipal 774 / 1975,7.0,REGIONAL PORTÃO,2.123845e+06,7068.306546,"POLYGON ((671332.905 7184743.04, 671376.041 71..."
1,112,23.0,DIVISA DE BAIRROS,GUABIROTUBA,Decreto Municipal 774 / 1975,3.0,REGIONAL CAJURU,2.654431e+06,6576.157173,"POLYGON ((676773.573 7181043.613, 676773.533 7..."
2,113,27.0,DIVISA DE BAIRROS,PORTÃO,Decreto Municipal 774 / 1975,7.0,REGIONAL PORTÃO,5.858496e+06,10839.074473,"POLYGON ((671803.064 7182533.926, 671788.777 7..."
3,115,45.0,DIVISA DE BAIRROS,MOSSUNGUÊ,Decreto Municipal 774 / 1975,5.0,REGIONAL SANTA FELICIDADE,3.365441e+06,9125.525056,"POLYGON ((669136.262 7186337.442, 669520.266 7..."
4,116,24.0,DIVISA DE BAIRROS,PRADO VELHO,Decreto Municipal 774 / 1975,1.0,REGIONAL MATRIZ,2.434125e+06,7089.468068,"POLYGON ((676031.648 7184686.502, 676114.557 7..."


In [ ]:
len(bairros)

75

In [ ]:
bairros.columns

Index(['OBJECTID', 'CODIGO', 'TIPO', 'NOME', 'FONTE', 'CD_REGIONA',
       'NM_REGIONA', 'SHAPE_AREA', 'SHAPE_LEN', 'geometry'],
      dtype='object')

In [ ]:
bairro = bairros[bairros['NOME'].str.contains('REBO', case=False)]

In [ ]:
print(bairro)
print(len(bairro))

    OBJECTID  CODIGO               TIPO      NOME  \
58       137     8.0  DIVISA DE BAIRROS  REBOUÇAS   

                           FONTE  CD_REGIONA       NM_REGIONA    SHAPE_AREA  \
58  Decreto Municipal 774 / 1975         1.0  REGIONAL MATRIZ  2.966143e+06   

      SHAPE_LEN                                           geometry  
58  7556.378992  POLYGON ((675792.743 7184707.24, 675819.582 71...  
1


In [ ]:
adjacentes = bairros[bairros.touches(bairro.unary_union)]

/tmp/ipykernel_10378/1514177483.py:1: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  adjacentes = bairros[bairros.touches(bairro.unary_union)]


In [ ]:
bairros = bairros.to_crs(epsg=4326)
bairro = bairro.to_crs(epsg=4326)
adjacentes = adjacentes.to_crs(epsg=4326)

In [ ]:
import folium

mapa = folium.Map(location=[-25.43, -49.27], zoom_start=13)

# Rebouças (laranja)
folium.GeoJson(
    bairro,
    style_function=lambda x: {
        'color': 'orange',
        'fillColor': 'orange',
        'fillOpacity': 0.6
    }
).add_to(mapa)

# Adjacentes (verde)
folium.GeoJson(
    adjacentes,
    style_function=lambda x: {
        'color': 'green',
        'fillColor': 'green',
        'fillOpacity': 0.4
    }
).add_to(mapa)

mapa

In [ ]:
bairro_utm = bairro.to_crs(epsg=31982)
area = bairro_utm.geometry.area.values[0]

print("Área em m²:", area)

Área em m²: 2966144.1855234657


In [ ]:
from shapely.geometry import Point

ponto = Point(-49.27, -25.43)

bairro_ponto = bairros[bairros.contains(ponto)]

bairro_ponto['NOME']

,NOME


In [ ]:
# Versão inicial do dropdown
# Nesta etapa, o dropdown apenas permite selecionar um bairro, mostrando o bairro escolhido e seus adjacentes destacados.
import ipywidgets as widgets
from IPython.display import display
import folium

dropdown = widgets.Dropdown(
    options=sorted(bairros['NOME'].unique()),
    description='Bairro:'
)

def atualizar_mapa(nome_bairro):
    bairro_sel = bairros[bairros['NOME'] == nome_bairro]
    adj = bairros[bairros.touches(bairro_sel.unary_union)]

    mapa = folium.Map(location=[-25.43, -49.27], zoom_start=13)

    # bairro escolhido
    folium.GeoJson(
        bairro_sel,
        style_function=lambda x: {
            'color': 'orange',
            'fillColor': 'orange',
            'fillOpacity': 0.6
        }
    ).add_to(mapa)

    # adjacentes
    folium.GeoJson(
        adj,
        style_function=lambda x: {
            'color': 'green',
            'fillColor': 'green',
            'fillOpacity': 0.4
        }
    ).add_to(mapa)

    display(mapa)

# conectar dropdown à função
widgets.interact(atualizar_mapa, nome_bairro=dropdown)

interactive(children=(Dropdown(description='Bairro:', options=('ABRANCHES', 'AHÚ', 'ALTO BOQUEIRÃO', 'ALTO DA …

<function __main__.atualizar_mapa(nome_bairro)>

In [ ]:
from shapely.geometry import Point
import folium

ponto = Point(-49.27, -25.43)

bairro_ponto = bairros[bairros.contains(ponto)]

mapa = folium.Map(location=[-25.43, -49.27], zoom_start=13)

# ponto
folium.Marker(
    location=[-25.43, -49.27],
    popup="Ponto"
).add_to(mapa)

# bairro encontrado
folium.GeoJson(
    bairro_ponto,
    style_function=lambda x: {
        'color': 'green',
        'fillColor': 'green'
    }
).add_to(mapa)

mapa

print(bairro_ponto['NOME'])

Series([], Name: NOME, dtype: object)


In [ ]:
# Escolas Municipais baixadas no IPPUC
import geopandas as gpd

caminho_escolas = '/content/drive/MyDrive/Colab Notebooks/Aula04/ESCOLA_MUNICIPAL_SIRGAS/ESCOLA_MUNICIPAL.shp'

escolas = gpd.read_file(caminho_escolas)

escolas.head()

,CD_EQUI,CD_LOCAL,CD_TEMA,TEMA,ID_EQUIP,EQUIPAMENT,CD_TIPO_EQ,TIPO_EQUI,CD_DEP_ADM,DEP_ADMIN,...,DT_ATUALIZ,OBSERVACAO,FONTE,CD_MANTENE,DS_MANTENE,SIGLA_MANT,INATIVO_EQ,LAT_SIRGAS,LON_SIRGAS,geometry
0,10880,41149378,11,EDUCAÇÃO,22,Escola,1,Ensino Fundamental,3,Público Municipal,...,2024-03-08,"Atendimento:\r\n-2ª a 6ª feira, das 7h30 às 11...",Secretaria Municipal da Educação,13,Secretaria Municipal da Educação,SME,0,-25.436154,-49.281744,POINT (672788.093 7185643.072)
1,5315,41537904,11,EDUCAÇÃO,22,Escola,1,Ensino Fundamental,3,Público Municipal,...,2024-03-08,Junto à escola funciona a Biblioteca Escolar B...,Secretaria Municipal da Educação,13,Secretaria Municipal da Educação,SME,0,-25.418166,-49.263690,POINT (674629.845 7187612.05)
2,8417,None,11,EDUCAÇÃO,22,Escola,1,Ensino Fundamental,3,Público Municipal,...,2024-07-08,Equipamento em fase de implantação.\r\nUE-MZ-8,Secretaria Municipal da Educação,13,Secretaria Municipal da Educação,SME,0,-25.432233,-49.241053,POINT (676886.546 7186024.039)
3,5314,41537890,11,EDUCAÇÃO,22,Escola,1,Ensino Fundamental,3,Público Municipal,...,2024-09-16,Junto à escola funciona a Biblioteca Escolar P...,Secretaria Municipal da Educação,13,Secretaria Municipal da Educação,SME,0,-25.441716,-49.258020,POINT (675166.21 7184996.069)
4,1620,41133323,11,EDUCAÇÃO,22,Escola,1,Ensino Fundamental,3,Público Municipal,...,2024-03-19,Junto à escola funciona a Biblioteca Escolar M...,Secretaria Municipal da Educação,13,Secretaria Municipal da Educação,SME,0,-25.452702,-49.281712,POINT (672767.694 7183810.086)


In [ ]:
escolas = escolas.to_crs(bairros.crs)

In [ ]:
escolas_no_bairro = escolas[escolas.within(bairro.union_all())]

print(escolas_no_bairro)

   CD_EQUI  CD_LOCAL  CD_TEMA      TEMA  ID_EQUIP EQUIPAMENT  CD_TIPO_EQ  \
3     5314  41537890       11  EDUCAÇÃO        22     Escola           1   

            TIPO_EQUI  CD_DEP_ADM          DEP_ADMIN  ... DT_ATUALIZ  \
3  Ensino Fundamental           3  Público Municipal  ... 2024-09-16   

                                          OBSERVACAO  \
3  Junto à escola funciona a Biblioteca Escolar P...   

                              FONTE CD_MANTENE  \
3  Secretaria Municipal da Educação         13   

                         DS_MANTENE SIGLA_MANT INATIVO_EQ LAT_SIRGAS  \
3  Secretaria Municipal da Educação        SME          0 -25.441716   

  LON_SIRGAS                       geometry  
3  -49.25802  POINT (675166.21 7184996.069)  

[1 rows x 51 columns]


In [ ]:
for _, row in escolas_no_bairro.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup="Escola"
    ).add_to(mapa)

In [ ]:
escolas.columns

Index(['CD_EQUI', 'CD_LOCAL', 'CD_TEMA', 'TEMA', 'ID_EQUIP', 'EQUIPAMENT',
       'CD_TIPO_EQ', 'TIPO_EQUI', 'CD_DEP_ADM', 'DEP_ADMIN', 'NOME_COMPL',
       'PRE_NOME', 'NOME', 'SIGLA_EQUI', 'CONVENIADO', 'NOME_ABREV',
       'NOME_MAPA', 'CD_RUA', 'NOME_RUA', 'NOME_RUANO', 'NUM_PRED',
       'COMPL_END', 'DIVULGAR_E', 'INDFISCAL', 'CD_BAIRRO', 'BAIRRO',
       'QUADR_EQUI', 'CD_REGIONA', 'REGIONAL', 'CD_DISTRIT', 'CD_EQUI_DI',
       'NM_DISTRIT', 'FUNC_MANHA', 'FUNC_TARDE', 'FUNC_NOITE', 'FUNC_24HR',
       'TELEFONE', 'RAMAL', 'EMAIL', 'SITE', 'DT_INAUGUR', 'DT_ATUALIZ',
       'OBSERVACAO', 'FONTE', 'CD_MANTENE', 'DS_MANTENE', 'SIGLA_MANT',
       'INATIVO_EQ', 'LAT_SIRGAS', 'LON_SIRGAS', 'geometry'],
      dtype='object')

In [ ]:
for _, row in escolas_no_bairro.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=row['NOME']
    ).add_to(mapa)

In [ ]:
# Versão final via dropdown
import ipywidgets as widgets
from IPython.display import display
import folium

dropdown = widgets.Dropdown(
    options=sorted(bairros['NOME'].unique()),
    description='Bairro:'
)

def atualizar_mapa(nome_bairro):
    # selecionar bairro
    bairro_sel = bairros[bairros['NOME'] == nome_bairro]

    # adjacentes
    adj = bairros[bairros.touches(bairro_sel.unary_union)]

    # escolas dentro do bairro
    escolas_no_bairro = escolas[escolas.within(bairro_sel.unary_union)]

    # criar novo mapa
    mapa = folium.Map(location=[-25.43, -49.27], zoom_start=13)

    # bairro (laranja)
    folium.GeoJson(
        bairro_sel,
        style_function=lambda x: {
            'color': 'orange',
            'fillColor': 'orange',
            'fillOpacity': 0.6
        },
        tooltip=folium.GeoJsonTooltip(fields=['NOME'])
    ).add_to(mapa)

    # adjacentes (verde)
    folium.GeoJson(
        adj,
        style_function=lambda x: {
            'color': 'green',
            'fillColor': 'green',
            'fillOpacity': 0.4
        }
    ).add_to(mapa)

    # escolas (marcadores)
    for _, row in escolas_no_bairro.iterrows():
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=str(row.iloc[0])  # depois podemos ajustar o nome
        ).add_to(mapa)

    display(mapa)

widgets.interact(atualizar_mapa, nome_bairro=dropdown)

interactive(children=(Dropdown(description='Bairro:', options=('ABRANCHES', 'AHÚ', 'ALTO BOQUEIRÃO', 'ALTO DA …

<function __main__.atualizar_mapa(nome_bairro)>